In [ ]:
# Imports
from functools import partial
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import eigsh
import jax
import jax.numpy as jnp
from flax import linen as nn
import optax
jax.config.update('jax_enable_x64', True)
plt.style.use('dark_background')
plt.rcParams.update({'axes.labelsize': 20, 'axes.titlesize': 20, 'legend.fontsize': 20,
                     'xtick.labelsize': 20, 'ytick.labelsize': 20})


In [ ]:
# Parameters
sizes = (12, 16)
couplings = {n: (0., .25, .5, .75) for n in sizes}
seeds = (11, 29, 47)
hidden, steps, batch, learning_rate = 16, 3600, 100, .005
swap_pairs, repeats, swap_batch = (2000, 8000, 32000), 10, 2000


In [ ]:
# RNN wavefunction
class RNN(nn.Module):
    hidden: int

    def setup(self):
        self.cell = nn.GRUCell(name='gru_cell', features=self.hidden,
                               kernel_init=jax.nn.initializers.glorot_uniform(), param_dtype=jnp.float64)
        self.rnn = nn.RNN(self.cell, return_carry=True)
        self.dense = nn.Dense(2, name='dense_layer',
                              kernel_init=jax.nn.initializers.glorot_uniform(), param_dtype=jnp.float64)

    def __call__(self, samples):
        onehot = jax.nn.one_hot(samples.astype(int), 2, dtype=jnp.float64)
        inputs = jnp.roll(onehot, 1, axis=1).at[:, 0].set(0.)
        _, output = self.rnn(inputs)
        return jnp.sum(nn.log_softmax(self.dense(output)) * onehot, axis=(1, 2))

    def sample(self, key, count, n):
        inputs = jnp.zeros((count, 2), dtype=jnp.float64)
        carry = self.cell.initialize_carry(jax.random.key(1), inputs.shape)
        samples = jnp.zeros((count, n), dtype=jnp.float64)
        logp = jnp.zeros(count, dtype=jnp.float64)
        for site, site_key in enumerate(jax.random.split(key, n)):
            carry, output = self.cell(carry, inputs)
            logits = self.dense(output)
            bit = jax.random.categorical(site_key, logits)
            inputs = jax.nn.one_hot(bit, 2, dtype=jnp.float64)
            samples = samples.at[:, site].set(bit)
            logp += jnp.sum(nn.log_softmax(logits) * inputs, axis=1)
        return samples, logp


model = RNN(hidden)
optimizer = optax.adam(learning_rate)
log_probability = jax.jit(lambda p, s: model.apply(p, s))


@partial(jax.jit, static_argnums=(2, 3))
def sample(params, key, n, count):
    return model.apply(params, key, count, n, method='sample')


In [ ]:
# ANNNI Hamiltonian
def hamiltonian(n, kappa):
    index = np.arange(1 << n)
    bits = ((index[:, None] >> np.arange(n-1, -1, -1)) & 1).astype(float)
    z = 1-2*bits
    diagonal = -np.sum(z[:, :-1]*z[:, 1:], axis=1)+kappa*np.sum(z[:, :-2]*z[:, 2:], axis=1)
    columns = np.concatenate([index]+[index ^ (1 << i) for i in range(n)])
    values = np.concatenate([diagonal]+[-np.ones(len(index)) for _ in range(n)])
    h = sparse.coo_matrix((values, (np.tile(index, n+1), columns)), shape=(len(index), len(index))).tocsr()
    return h, bits


# Renyi-2 entropy
def entropy(psi, n):
    return np.array([-np.log2(np.sum(np.linalg.svd(psi.reshape(1 << cut, -1), compute_uv=False)**4))
                     for cut in range(1, n)])


# Wavefunction
def wavefunction(params, bits):
    logp = np.concatenate([np.asarray(log_probability(params, jnp.asarray(bits[i:i+256])))
                           for i in range(0, len(bits), 256)])
    psi = np.exp(logp/2)
    return psi/np.linalg.norm(psi)


In [ ]:
# Local energy
def local_energy(params, samples, kappa):
    n = samples.shape[1]
    logp = model.apply(params, samples)
    flips = jax.vmap(lambda site: samples.at[:, site].set(1-samples[:, site]))(jnp.arange(n))
    flipped_logp = model.apply(params, flips.reshape(-1, n)).reshape(n, -1)
    spins = 1-2*samples
    diagonal = -jnp.sum(spins[:, :-1]*spins[:, 1:], axis=1) + kappa*jnp.sum(spins[:, :-2]*spins[:, 2:], axis=1)
    return diagonal-jnp.sum(jnp.exp(.5*(flipped_logp-logp)), axis=0)


# VMC update
@jax.jit
def update(params, state, draws, kappa):
    def loss(p):
        energy = jax.lax.stop_gradient(local_energy(p, draws, kappa))
        return jnp.mean(model.apply(p, draws)*(energy-energy.mean()))
    changes, state = optimizer.update(jax.grad(loss)(params), state, params)
    return optax.apply_updates(params, changes), state


# Training
def train(n, kappa, seed, h, bits, exact_energy):
    key, init_key = jax.random.split(jax.random.key(seed))
    params = model.init(init_key, jnp.zeros((2, n)))
    state = optimizer.init(params)
    trace = []
    for step in range(steps+1):
        if step % 200 == 0 or step == steps:
            psi = wavefunction(params, bits)
            trace.append((step, abs(float(psi@(h@psi))-exact_energy)/n))
        if step == steps:
            break
        key, draw_key = jax.random.split(key)
        draws, _ = sample(params, draw_key, n, batch)
        params, state = jax.block_until_ready(update(params, state, draws, jnp.asarray(kappa)))
    return params, np.array(trace)


In [ ]:
# SWAP estimator
@jax.jit
def swap_ratio(params, a, b, loga, logb):
    cut = a.shape[1]//2
    exchanged = jnp.concatenate((jnp.concatenate((b[:, :cut], a[:, cut:]), axis=1),
                                 jnp.concatenate((a[:, :cut], b[:, cut:]), axis=1)))
    logp = model.apply(params, exchanged).reshape(2, -1)
    return jnp.exp(.5*(logp.sum(axis=0)-loga-logb))


# Entropy sampling
def sampled_entropy(params, n, tag):
    estimates = np.empty((repeats, len(swap_pairs)))
    for repeat in range(repeats):
        key = jax.random.fold_in(jax.random.key(7000003+repeat), tag)
        ratios = np.empty(swap_pairs[-1])
        for start in range(0, swap_pairs[-1], swap_batch):
            key, ka, kb = jax.random.split(key, 3)
            a, la = sample(params, ka, n, swap_batch)
            b, lb = sample(params, kb, n, swap_batch)
            ratios[start:start+swap_batch] = np.asarray(swap_ratio(params, a, b, la, lb))
            count = start+swap_batch
            if count in swap_pairs:
                estimates[repeat, swap_pairs.index(count)] = -np.log2(ratios[:count].mean())
    return estimates


In [ ]:
# Ground-state calculation
results = []
print('N  kappa seed  E/N        dE/N      S2_ED    S2_NQS   S2_SWAP')
for n in sizes:
    for kappa in couplings[n]:
        h, bits = hamiltonian(n, kappa)
        eigenvalues, eigenvectors = eigsh(h, k=1, which='SA', tol=1e-12, maxiter=20000,
                                          v0=np.random.default_rng(829).normal(size=1 << n))
        exact_energy, exact_psi = float(eigenvalues[0]), eigenvectors[:, 0]
        exact_s2 = entropy(exact_psi, n)
        for seed in seeds:
            params, trace = train(n, kappa, seed, h, bits, exact_energy)
            psi = wavefunction(params, bits)
            energy = float(psi@(h@psi))
            s2 = entropy(psi, n)
            estimates = sampled_entropy(params, n, len(results))
            energy_error = abs(energy-exact_energy)/n
            entropy_error = abs(s2[n//2-1]-exact_s2[n//2-1])
            results.append(dict(n=n, kappa=kappa, seed=seed, energy=energy, S2=s2, exact_S2=exact_s2,
                                energy_error=energy_error, entropy_error=entropy_error, trace=trace, swap=estimates))
            print(f'{n:2} {kappa:5.2f} {seed:4} {energy/n:.6f} {energy_error:.2e} '
                  f'{exact_s2[n//2-1]:.6f} {s2[n//2-1]:.6f} {estimates[:, -1].mean():.6f}')


In [ ]:
# Training convergence
colors = ('r', '#4da6ff', 'g', 'c')
fig, axes = plt.subplots(1, len(sizes), figsize=(20, 5), layout='constrained', squeeze=False, sharex=True, sharey=True)
for ax, n in zip(axes[0], sizes):
    for kappa, color in zip(couplings[n], colors):
        group = [r for r in results if r['n'] == n and r['kappa'] == kappa]
        x = group[0]['trace'][:, 0]
        y = np.array([r['trace'][:, 1] for r in group])
        for curve in y:
            ax.semilogy(x, curve, color=color, alpha=.25, linewidth=1)
        ax.semilogy(x, y.mean(axis=0), '--o', color=color, markersize=4, label=rf'$\kappa={kappa:g}$')
    ax.set_xticks(np.linspace(0, steps, 7, dtype=int))
    ax.set(xlabel='Training step', ylabel=r'$|E-E_{\rm ED}|/N$', title=f'Training, N={n}')
    ax.legend(loc='best', fontsize=16)
plt.show()

# Final errors
fig, axes = plt.subplots(1, 2, figsize=(20, 5), layout='constrained')
for n, color, marker in zip(sizes, ('r', 'c'), ('o', 's')):
    kappas = couplings[n]
    groups = [[r for r in results if r['n'] == n and r['kappa'] == k] for k in kappas]
    axes[0].semilogy(kappas, [max(r['energy_error'] for r in g) for g in groups],
                     '--', marker=marker, color=color, label=f'N={n}')
    axes[1].semilogy(kappas, [max(r['entropy_error'] for r in g) for g in groups],
                     '--', marker=marker, color=color, label=f'N={n}')
axes[0].set(xlabel=r'$\kappa$', ylabel=r'$\max_{\rm seed}|E-E_{\rm ED}|/N$', title='Final energy error')
axes[1].set(xlabel=r'$\kappa$', ylabel=r'$\max_{\rm seed}|S_2-S_{2,\rm ED}|$ (bits)', title='Final entropy error')
for ax in axes:
    ax.legend(loc='best', fontsize=16)
plt.show()

# Entanglement
fig, axes = plt.subplots(1, len(sizes), figsize=(20, 5), layout='constrained', squeeze=False, sharex=True, sharey=True)
for ax, n in zip(axes[0], sizes):
    kappas = np.array(couplings[n])
    groups = [[r for r in results if r['n'] == n and r['kappa'] == k] for k in kappas]
    half = n//2-1
    exact = [g[0]['exact_S2'][half] for g in groups]
    ax.plot(kappas, exact, '--o', color='w', markerfacecolor='none', markersize=9, label='ED')
    values = np.array([[r['S2'][half] for r in g] for g in groups])
    mean = values.mean(axis=1)
    ax.errorbar(kappas, mean, yerr=[mean-values.min(axis=1), values.max(axis=1)-mean],
                fmt='xr', markersize=10, capsize=4, label='NQS: mean and seed range')
    values = [np.concatenate([r['swap'][:, -1] for r in g]) for g in groups]
    mean = np.array([v.mean() for v in values])
    ax.errorbar(kappas, mean, yerr=[mean-[v.min() for v in values], [v.max() for v in values]-mean],
                fmt='sc', markerfacecolor='none', markersize=9, capsize=4, label='SWAP: mean and repeat range')
    ax.set(xlabel=r'$\kappa$', ylabel=r'$S_2$ (bits)', title=f'Entanglement, N={n}')
    ax.legend(loc='best', fontsize=15)
plt.show()

# Sampling error
fig, axes = plt.subplots(1, len(sizes), figsize=(20, 5), layout='constrained', squeeze=False, sharex=True, sharey=True)
for ax, n in zip(axes[0], sizes):
    for kappa, color in zip(couplings[n], colors):
        group = [r for r in results if r['n'] == n and r['kappa'] == kappa]
        error = np.concatenate([r['swap']-r['S2'][n//2-1] for r in group])
        ax.loglog(swap_pairs, np.sqrt(np.mean(error**2, axis=0)), '--o', color=color, label=rf'$\kappa={kappa:g}$')
    ax.set(xlabel='Independent sample pairs', ylabel='RMSE (bits)', title=f'Sampling error, N={n}')
    ax.set_xticks(swap_pairs, labels=[str(x) for x in swap_pairs])
    ax.set_xticks([], minor=True)
    ax.legend(loc='best', fontsize=16)
plt.show()

print('N   Pairs   Sampling RMSE (bits)')
for n in sizes:
    measured = [r for r in results if r['n'] == n]
    for i, count in enumerate(swap_pairs):
        error = np.concatenate([r['swap'][:, i]-r['S2'][n//2-1] for r in measured])
        print(f'{n:2} {count:6}   {np.sqrt(np.mean(error**2)):.6f}')
